# Prepare Training Data for YOLOv5
This Jupyter notebook should be used to prepare data for training YOLOv5. The mask annotations (for different instances of objects) are  used to create the training data in the following steps:
- Read the mask annotations and convert them to bounding boxes. This notebook support reading and parsing training data generated by the CellPose model and also by the annotators.
- Crop the large images to smaller sub-images to keep the bounding box sizes large enough for the model to be able to detect them. 
- Convert the format of the annotations to the format expected by YOLOv5. 

In [ ]:
# load required libraries
import os
from PIL import Image
from IPython.display import display
import numpy as np
import shutil
import cv2
import pandas as pd
from typing import List, Union, Dict, Final

## YOLOv5 Configurations
The following cell specified the base path for storing the parsed train/test images and annotations, as well as where the config yaml file for YOLOv5 can be found. This file should be created in advance. It should include the following info:
- The paths (relative to the specified base path above) for storing the train/test data (images and annotations)
- The number of classes
- The classnames

The `RESIZE_FLAG` is only used for preparing training data from the images annotated by the annotators. If set to True, the images are resized by factor = 1/2 before cropping to smaller sub-images. 

In [ ]:
# output (base) path where the images and labels folders 
# should be created for YOLOv4 
OUTPUT_BASE_PATH = os.getcwd()
# the config file (including the path with respect to the base path above) containing
# the classnames and also the location where the training data should be created
OUTPUT_CONFIG_FILENAME = 'data/caging_analysis_cells.yaml' 
# YOLO model input size (square)
YOLOV5_INPUT_SIZE = 640
# A flag to indicate the training images should be scaled down by a factor of 2 before cropping
RESIZE_FLAG = True

### Parse the yaml config file

In [ ]:
# open the config file
config_file = open(os.path.join(OUTPUT_BASE_PATH, OUTPUT_CONFIG_FILENAME), 'r')

# extract the 'path' specified in the config
yaml_path_param = os.getcwd()
path_found = False
for i, row in enumerate(config_file):
    row_details = row.split(':')
    if row_details[0].strip() == 'path':
        yaml_path_param = row_details[1].strip()
        print('[INFO]: Path specified in the yaml file: ', yaml_path_param)
        path_found=True
        break
        
if not path_found:
    print('[ERROR]: No path parameter was found in the config file! Data will be stored in the current folder!')

# extract the paths for the train and test data specified in the config
for i, row in enumerate(config_file):
    row_details = row.split(':')
    if row_details[0].strip() == 'train':
        TRAIN_IMAGE_FOLDER = os.path.join(yaml_path_param, row_details[1].strip())
    if row_details[0].strip() == 'val':
        TEST_IMAGE_FOLDER = os.path.join(yaml_path_param, row_details[1].strip())
    if row_details[0].strip() == 'names':
        classnames = eval(row_details[1].strip())
        REVERSE_LABEL_MAP = {classnames[i]: i for i in range(len(classnames))}
        LABEL_MAP = {i:classnames[i] for i in range(len(classnames))}
    if row_details[0].strip() == 'nc':
        num_classes = int(row_details[1].strip())

config_file.close()

if num_classes != len(classnames):
    print('[ERROR]: The number of classes in \'nc\' does not match with the list of classes in \'names\'')
else:
    print('[INFO]: Mapping between names and labels: ', REVERSE_LABEL_MAP)
    print('[INFO]: Make sure each class is labeled with the specified index in the annotated mask!')

### Create the output folders

In [ ]:
folders = TRAIN_IMAGE_FOLDER.split('/')
if 'images' in folders:
    
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Train output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
    
    # replace the "images" in the train/test images folder with "labels" to create the path
    # for the annotations
    ind = len(folders) - [f for f in reversed(folders)].index('images') - 1
    folders[ind] = 'labels'
    
    TRAIN_LABEL_FOLDER = '/'.join(folders)
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Train output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
        
else:
    print('[ERROR]: Invalid name for training folder in \'train\': It should include word \'images\'')


folders = TEST_IMAGE_FOLDER.split('/')
if 'images' in folders:
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Test output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
    
    # replace the "images" in the train/test images folder with "labels" to create the path
    # for the annotations
    ind = len(folders) - [f for f in reversed(folders)].index('images') - 1
    folders[ind] = 'labels'
    
    TEST_LABEL_FOLDER = '/'.join(folders)
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Test output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
else:
    print('[ERROR]: Invalid name for test folder in \'val\': It should include word \'images\'')

print('Train images folder: %s' %TRAIN_IMAGE_FOLDER)
print('Train labels folder: %s' %TRAIN_LABEL_FOLDER)
print('Test images folder: %s' %TEST_IMAGE_FOLDER)
print('Test labels folder: %s' %TEST_LABEL_FOLDER)

## Prepare Data

Two options are provided below for preparing the training data using
1. The mask annotations generated by CellPose 
2. Image annotations from the annotators

Use only one option. 
### 1. Data model for mask annotations generated by CellPose (one-class)
Note that the following class can only parse the masks from CellPose. This parser only works for One class. Use the updated class below for parsing annotations by the annotators. 

In [ ]:
# a similar class to pytorch dataset to "parse" the masks and extract the bounding boxes
# but in YOLO format, which is (center_x, center_y, w, h) each normalized to the image's width/height
class CellMaskDataset:
    def __init__(self, images_path: str, masks_path: str, 
                 color_depth: int = 13, 
                 normalize:bool = False) -> None:
        self.images_path = images_path
        self.masks_path = masks_path
        # load all images and masks
        # the assumption is the image and its mask annotation use the same name
        self.imgs = list(sorted(os.listdir(images_path)))
        self.masks = list(sorted(os.listdir(masks_path)))
        # the scaling factor for normalizing the channels after Tensor
        # conversion to get values in [0, 1]
        # this is 2 ^ color_depth - 1, where color_depth is the number of bits
        # use to represent the intensities for each channel
        self.channel_scale = 2 ** color_depth - 1
        self.normalize = normalize
        
        if len(self.imgs) != len(self.masks):
            print("[ERROR]: The list of images and masks are not consistent")
            return
        
        for i, img_filename in enumerate(self.imgs):
            # drop the image/mask filename extension 
            # (anything after the last '.' in the filename is considered as extension)
            img_name = ".".join(img_filename.strip().split('.')[:-1])
            mask_name = ".".join(self.masks[i].strip().split('.')[:-1])
            if img_name != mask_name:
                print("[ERROR]: Inconsistent mask file :{} found for image file: {}".format(mask_name, img_name))
     

    def __getitem__(self, idx: int):
        # load images and masks
        img_path = os.path.join(self.images_path, self.imgs[idx])
        mask_path = os.path.join(self.masks_path, self.masks[idx])
        # read the image, do not change the format
        # depending on the set color_depth, the values will be in [0, 2^color_depth - 1]
        # img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        img = np.array(Image.open(img_path))
        image_height, image_width = img.shape[:2]
        
        # the assumption here is instances are encoded as different levels
        # with 0 being the background
        # each element in mask is an np.uint16 (unsigned 16 bits)
        # we can read such images using PIL.Image
        # convert to a numpy array
        mask = np.array(Image.open(mask_path))
        
        
        # instances are encoded as different gray levels
        # the results are sorted
        obj_ids = np.unique(mask)
        # first id [0] is the background, so remove it
        obj_ids = obj_ids[1:]
        
        # get bounding box coordinates for each mask
        num_objs = len(obj_ids)
        
        # split the color-encoded mask into a set
        # of binary masks
        masks = mask == obj_ids[:, None, None]
        
        boxes = []
        labels = []
        
        for i in range(num_objs):
            
            pos = np.where(masks[i])
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            if xmin < xmax and ymin < ymax:
                boxes.append([xmin, ymin, xmax, ymax])
                # add the label, we support only one class for now
                # note that YOLO labels start with 0, and the mask
                # labels start with 1, so we need to subtract 1 here 
                # also, we are saving the classname here
                labels.append(LABEL_MAP[0])
        
        
        # create a pandas DataFrame for ease of procesing
        annotations_df = pd.DataFrame(columns=['xtl', 'ytl', 'xbr', 'ybr', 'label'])
        annotations_df[['xtl', 'ytl', 'xbr', 'ybr']] = boxes
        annotations_df['label'] = labels

        # convert the returned np.unit32 image to float with values between 0, 1
        if self.normalize:
            # if this flag is set, normalize the image such the the minimum intensity 
            # is mapped to zero, and the maximum is mapped to one
             # convert the image to a numpy array
            img = cv2.normalize(img, img, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX).astype(np.uint8)
        else:    
            # the intensity of the images is color_deptj bits, so we need to divide by 2^color_depth - 1
            img = (255 * img / self.channel_scale).astype(np.uint8)
            
        return  {'name': self.imgs[idx], 'image': img, 'annotations': annotations_df}

    def __len__(self):
        return len(self.imgs)

#### CellPose dataset classes

In [ ]:
TRAIN_IMAGES_PATH = '/home/cellareye/Cellanome/Images/img/images'
TRAIN_MASKS_PATH = '/home/cellareye/Cellanome/Images/img/masks'

TEST_IMAGES_PATH = '/home/cellareye/Cellanome/Images/img/test/images'
TEST_MASKS_PATH = '/home/cellareye/Cellanome/Images/img/test/masks'

# use our dataset and defined transformations
train_dataset = CellMaskDataset(images_path=TRAIN_IMAGES_PATH, masks_path=TRAIN_MASKS_PATH, 
                                color_depth = 14, normalize=False)

test_dataset = CellMaskDataset(images_path=TEST_IMAGES_PATH, masks_path=TEST_MASKS_PATH, 
                               color_depth = 14, normalize=False)

### 2. Data model for annotated data by the annotators (multi-class)
Note that the data model class below returns masks as well, but it is not used here. 

In [ ]:
import sys
sys.path.append('../utils')
from json_parser import CellMaskDataset, optimize_crop, crop_and_block, show_sample

#### Dataset for annotated images

In [ ]:
# the path to all annotation files
# ANNOTATIONS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/Json'
ANNOTATIONS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/Json'
# this folder will include all images (train or test)
# if an image is missing, it will get downloaded from the url specified in the annotation file
# the image names should be the same as the annotation file names
IMAGES_PATH = '/home/cellareye/Cellanome/Data/Images'

# TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/train'
# TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/test'
TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/train'
TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/test'

#### Randomly split the data to train/test sets (if not already done)
Skip the cell below if the train and test sets are already created in the folders specified above! Otherwise, run the cell below to randomly select 10% of the annotated images as test images.

In [ ]:
SEED = 7
import random
import shutil
annotation_files = os.listdir(ANNOTATIONS_PATH)
random.seed(SEED)
random.shuffle(annotation_files)

if len(os.listdir(TRAIN_ANNOTS_PATH)) == 0 and len(os.listdir(TEST_ANNOTS_PATH)) == 0:

    for idx, file in enumerate(annotation_files):
        if idx < 0.1 * len(annotation_files):
            target_folder = TEST_ANNOTS_PATH
        else:
            target_folder = TRAIN_ANNOTS_PATH 
    
        shutil.copy(os.path.join(ANNOTATIONS_PATH, file), 
                    os.path.join(target_folder, file))
else:
    print('Train or test set already includes files')

#### Classes

In [ ]:
# use color depth = 8 for images shared with annotators; these are already processed images
# also, we resize the microscope images (with resolution 2000x1600)  and breadboard images 
# (with resolution 4512x4512) by a factor of 2 to 1000x800 and 2256x2256, respectively
# this allows us to train the model on a smaller objects and faster runtime
# however, we do not do it on images with resolution other than 2000x1600 (not from the microscope ) 
# as we do not have much idea about the size of cells in those images
# here is the size distribution of the set for the caging model
# (1600, 2000): 1201, (1944, 2592): 41, (2208, 2758): 69, (2208, 2756): 15
#  and here is the size distribution of the set for the analysis model
# (1600, 2000): 1377, (4512, 4512): 313
# we do not do the resizing here, but later with a function identical to
# the class below

# modifying the REVERSE_LABEL_MAP only here to map 'dying/dead cells' from caging dataset to 'Cell'
modified_reverse_label_map = REVERSE_LABEL_MAP.copy()
modified_reverse_label_map['dying/dead cells'] = 0

# we also pass the reverse_label_map_mod keys (class names) in labels_of_interest to only consider 
# 'Cell', 'Bead' and 'dying/dead cells' classes and ignore 'Cluster' (from caging dataset) and 'cages' 
# (from analysis dataset)

train_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TRAIN_ANNOTS_PATH,
                                labels_of_interest = list(modified_reverse_label_map.keys()),
                                color_depth=8, max_larger_side = 5000, max_smaller_side = 5000,
                                normalize=False, class_names_to_ids_map=modified_reverse_label_map)

test_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TEST_ANNOTS_PATH,
                               labels_of_interest = list(modified_reverse_label_map.keys()), 
                               color_depth=8, max_larger_side = 5000, max_smaller_side = 5000, 
                               normalize=False, class_names_to_ids_map=modified_reverse_label_map)

#### A helper class for target resizing of micropscope images

In [ ]:
def resize_annotations(annotations, max_larger_side, max_smaller_side):
    image_height, image_width = annotations['image'].shape[:2]
    larger_side: int = max(image_width, image_height)
    smaller_side: int = min(image_width, image_height)
    if larger_side > max_larger_side or smaller_side > max_smaller_side:
        scale_factor: float = max(float(larger_side) / max_larger_side, float(smaller_side) / max_smaller_side)
        # for decimating an image, cv2.INTER_AREA is the preferred method (scale_factor is always > 1) 
        annotations['image'] = cv2.resize(annotations['image'], 
                                          (int(image_width / scale_factor), int(image_height / scale_factor)),
                                          interpolation = cv2.INTER_AREA)
        # update all the masks and annotations
        annotations['annotations'][['xtl', 'ytl', 'xbr', 'ybr']] = \
        annotations['annotations'][['xtl', 'ytl', 'xbr', 'ybr']].div(scale_factor).astype(int)
            
        for idx in range(len(annotations['masks'])):
            annotations['masks'][idx] = cv2.resize(annotations['masks'][idx], 
                                                   (int(image_width / scale_factor), 
                                                    int(image_height / scale_factor)), 
                                                   interpolation = cv2.INTER_NEAREST)
                
        
        # make sure no box width/height becomes zero after the resize
        annotations['annotations'].reset_index(inplace=True, drop=True)
        # only keep boxes with positive width and height
        annotations['annotations'] = annotations['annotations'][
            (annotations['annotations']['ybr'] - annotations['annotations']['ytl'] > 0) & 
            (annotations['annotations']['xbr'] - annotations['annotations']['xtl'] > 0)]
        
        annotations['masks'] = [annotations['masks'][i] for i in annotations['annotations'].index]
        annotations['annotations'].reset_index(inplace=True, drop=True)
    return annotations

## Putting everything together - Saving the parsed annotations and images
### Generate the train and test data
Run the cell below twice, with the train flag set to True and False to generate the train and test data for YOLOv5 training. This will take the annotated data from the train and test classes and generate training data expected by YOLOv5.

In [ ]:
def prepare_data(train=True):
    if train:
        set_desc = 'training'
        image_folder = TRAIN_IMAGE_FOLDER
        label_folder = TRAIN_LABEL_FOLDER
        data_set = train_dataset
    else:
        set_desc = 'testing'
        image_folder = TEST_IMAGE_FOLDER
        label_folder = TEST_LABEL_FOLDER
        data_set = test_dataset

    # keep all the labels in the model label map
    class_ids_of_interest = list(LABEL_MAP.keys())

    num_images = 0
    num_annotations = 0

    # the number of overlapping pixels between crops in each dimension
    # this is larger than the largest expected cell size (in fact, 3 times 
    # more than the expected size of the cells in breadboard images)

    overlap_in_x = 100
    overlap_in_y = 100

    # the step size for the starting point of each crop in x and y dimension
    crop_start_step_x = YOLOV5_INPUT_SIZE - overlap_in_x
    crop_start_step_y = YOLOV5_INPUT_SIZE - overlap_in_y

    # read the image and the annotations, then parse each
    for idx in range(len(data_set)):

        sample = data_set[idx]
        # image size
        image_height, image_width = sample["image"].shape[:2]

        if RESIZE_FLAG:
            # resize by a factor of two
            if image_height == 1600 and image_width == 2000:
                sample = resize_annotations(sample, 1000, 800)
                image_height, image_width = sample["image"].shape[:2]
            elif image_height == 4512 and image_width == 4512:
                sample = resize_annotations(sample, 2256, 2256)
                image_height, image_width = sample["image"].shape[:2]
                
        crop_count = 0
        # overlapping crops
        for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
            for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
                # crop coordinates
                xc_tl = x_start
                yc_tl = y_start
                xc_br = x_start + YOLOV5_INPUT_SIZE
                yc_br = y_start + YOLOV5_INPUT_SIZE
                # make sure we always crop the image with the given size
                # if we get to the boundaries, extend the crop
                # size inside the image to always get the same size crop
                # this is not really needed, but help with capturing more
                # annotations toward the low/right parts of the image
                if xc_br > image_width:
                    xc_br = image_width 
                    xc_tl = xc_br - YOLOV5_INPUT_SIZE
                if yc_br > image_height:
                    yc_br = image_height
                    yc_tl = yc_br - YOLOV5_INPUT_SIZE

                crop_coords = [xc_tl, yc_tl, xc_br, yc_br]

                # optimize the crop
                crop_coords = optimize_crop(sample["annotations"], crop_coords, 
                                            overlap_in_x - 1, 
                                            overlap_in_y - 1, 
                                            image_width, image_height, class_ids_of_interest)

                # crop and block the image
                cropped_sample =  crop_and_block(sample, crop_coords, labels_of_interest=class_ids_of_interest)

                # save the cropped image and the annotations for this image, for each image name
                # use _ crop_count
                img_name = ".".join(cropped_sample["name"].strip().split('.')[:-1])

                # save the image in jpg format
                crp_img_name = img_name + '_crp_' + str(crop_count) + '.jpg'
                cv2.imwrite(os.path.join(OUTPUT_BASE_PATH, image_folder, crp_img_name), cropped_sample["image"])
                num_images += 1

                crop_height, crop_width = cropped_sample["image"].shape[:2]
                # annotation txt file
                crp_annot_name = img_name + '_crp_' + str(crop_count) + '.txt'
                # create the YOLOv5 annotations txt file for the cropped image
                annotation_file = open(os.path.join(OUTPUT_BASE_PATH, label_folder, crp_annot_name), 'w+')

                for _, row in cropped_sample["annotations"].iterrows():
                    # skip labels that are not in the label map 
                    # (not really needed as we have already filtered this above)
                    if row['label'] not in LABEL_MAP:
                        continue
                    num_annotations += 1
                    # YOLOv5 annotation format
                    center_x = (row['xtl'] + row['xbr']) / (2.0 * crop_width)
                    center_y = (row['ytl'] + row['ybr']) / (2.0 * crop_height)
                    w = (row['xbr'] - row['xtl']) / float(crop_width)
                    h = (row['ybr'] - row['ytl']) / float(crop_height)
                    label = row['label']
                    annotation_file.write(' '.join([str(label), str(center_x), str(center_y), str(w), str(h)]) + '\n')

                annotation_file.close()

                crop_count += 1


    print('Created {} '.format(num_images) + 'images for ' + set_desc)
    print('Created {} '.format(num_annotations) + 'objects for ' + set_desc)

In [ ]:
prepare_data(True)

In [ ]:
prepare_data(False)

### A function to check the parsed data

In [ ]:
# checks on the results
def check_results (idx, train = True):
    if train:
        image_folder = TRAIN_IMAGE_FOLDER
        label_folder = TRAIN_LABEL_FOLDER
    else:
        image_folder = TEST_IMAGE_FOLDER
        label_folder = TEST_LABEL_FOLDER
    
    dataset_files = os.listdir(os.path.join(OUTPUT_BASE_PATH, image_folder))
    name = dataset_files[idx]
    
    # colors for displaying bounding boxes
    COLORS = [(0, 0, 255), (0, 255, 0), (255, 0, 0),
              (255, 0, 255), (0, 255, 255), (255, 255, 0)]
    # label map
    LABEL_MAP = {value: key for key, value in REVERSE_LABEL_MAP.items()}
    
    image = cv2.imread(os.path.join(OUTPUT_BASE_PATH, image_folder, name))
    H, W = image.shape[:2]
    print('The image sizes are (H, W) = %d, %d' %(H, W))
    # read the annotations file
    with open(os.path.join(OUTPUT_BASE_PATH, label_folder, name[:-4] + '.txt'),'r') as annot_file:
        
        print('Parsing annotation file %s' %(name[:-4] + '.txt'))
        num_annotations = 0
        for line in annot_file:
            num_annotations += 1
            fields = line.strip().split(' ')

            (label, center_x, center_y, w, h) = fields
            xtl = int((float(center_x) - float(w) / 2.0) * W)
            ytl = int((float(center_y) - float(h) / 2.0) * H)
            xbr = int((float(center_x) + float(w) / 2.0) * W)
            ybr = int((float(center_y) + float(h) / 2.0) * H)
            
            # convert the label to an integer from string
            label = int(label)
            text = LABEL_MAP[label]
            classIds = list(REVERSE_LABEL_MAP.values())
            if label not in classIds:
                print('Incorrect label was found %s' %label)
                # use black for incorrect label
                color = (0, 0, 0)
            else:
                color = COLORS[label]
                
            cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
            cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    return Image.fromarray(image[:, :, (2, 1, 0)])

In [ ]:
check_results(idx=100, train = False)

## Checking the annotations by the annotators

In [ ]:
ANNOTATIONS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/Json'
IMAGES_PATH = '/home/cellareye/Cellanome/Data/Images'

In [ ]:
# the downloaded images are already processed, so use 8 bits for the bit-depth
dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=ANNOTATIONS_PATH, 
                          color_depth=8, max_larger_side = 5000, max_smaller_side = 5000,
                          normalize=False, class_names_to_ids_map=None)

In [ ]:
# this is an example of complex_polygon
annots = dataset[3]
img = show_sample(annots)
display(Image.fromarray(img[:, :, (2, 1, 0)]))